# Transformer Training on Arithmetic Dataset

In [ ]:

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/transformer-project/output'
print(f'Checkpoints will be saved to: {SAVE_DIR}')

In [ ]:
import os

REPO_DIR = '/content/transformer-project'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/KKD2005/transformer-project.git {REPO_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

!pip install -q -r requirements.txt

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import sys, os, json
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'data'))

TOKENIZER_PATH = os.path.join(REPO_DIR, 'data', 'arithmetic_tokenizer.json')
CORPUS_PATH    = os.path.join(REPO_DIR, 'data', 'arithmetic_tokenizer_corpus.txt')
TRAIN_INPUT_PATH = "arithmetic_train.json"
TEST_INPUT_PATH = 'arithmetic_test.json'
    
with open(TRAIN_INPUT_PATH, 'r', encoding='utf-8') as f:
    raw_dataset = json.loads(TRAIN_INPUT_PATH)


tmp_ds = ArithmeticDataset(raw_dataset, tokenizer=None)
tmp_ds.create_tokenizer_txt(CORPUS_PATH)
print(f'Corpus written to {CORPUS_PATH}')

from tokenizers import ByteLevelBPETokenizer
tok = ByteLevelBPETokenizer()
tok.train(
    files=[CORPUS_PATH],
    vocab_size=8192,
    min_frequency=2,
    special_tokens=['<pad>', '<eos>', '<bos>', '<problem>', '</problem>',
                    '<reasoning>', '</reasoning>', '<answer>', '</answer>']
)
tok.save(TOKENIZER_PATH)
print(f'Tokenizer saved to {TOKENIZER_PATH}')

## Documenting the Dataset

In [ ]:
import sys, os, re
sys.path.insert(0, os.path.join('..', 'data'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

In [ ]:
train_data = json.loads(TRAIN_INPUT_PATH)
test_data = json.loads(TEST_INPUT_PATH)
print(f"Train examples : {len(train_data):,}")
print(f"Test  examples : {len(test_data):,}")


In [ ]:
if os.path.exists(TOKENIZER_PATH):
    from transformers import PreTrainedTokenizerFast
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_file=TOKENIZER_PATH,
        pad_token='<pad>',
        eos_token='<eos>',
        bos_token='<bos>',
    )
    def tokenize(text):
        return tokenizer.encode(text, add_special_tokens=False)
    print(f"vocab size: {tokenizer.vocab_size:,}")
else:
    tokenizer = None
    def tokenize(text):
        return text.split()

In [ ]:
def format_example(row):
    """Reproduce the exact format used in GSM8KDataset."""
    reasoning, answer = row['reasoning'], row['answer']
    prefix = '<problem>' + row['problem'] + '</problem>\n'
    
    text   = prefix + '<reasoning>\n' + reasoning + '</reasoning>\n<answer>' + answer + '</answer>'
    return text, row['problem'], reasoning, answer

records = []
for row in train_data:
    text, question, reasoning, answer = format_example(row)
    tokens = tokenize(text)
    q_tokens = tokenize('<problem>' + question + '</problem>\n')
    r_tokens = tokenize('<reasoning>\n' + reasoning + '</reasoning>\n')
    a_tokens = tokenize('<answer>' + answer + '</answer>')
    records.append({
        'text': text,
        'question': question,
        'reasoning': reasoning,
        'answer': answer,
        'total_tokens': len(tokens),
        'q_tokens': len(q_tokens),
        'r_tokens': len(r_tokens),
        'a_tokens': len(a_tokens),
    })



In [ ]:
total_tokens = [r['total_tokens'] for r in records]
q_tokens     = [r['q_tokens']     for r in records]
r_tokens     = [r['r_tokens']     for r in records]
a_tokens     = [r['a_tokens']     for r in records]
for name, vals in [('question', q_tokens), ('reasoning', r_tokens), ('answer', a_tokens)]:
    print(f"  {name:<10}  mean={np.mean(vals):.1f}  median={np.median(vals):.1f}  max={max(vals)}")
MAX_LEN = 1024 
truncated = sum(1 for t in total_tokens if t > MAX_LEN)
print(f"\n  Examples exceeding max_length={MAX_LEN}: {truncated} ({truncated/len(records)*100:.1f}%)")
plt.hist(total_tokens)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Full example token distribution
ax = axes[0]
ax.hist(total_tokens, bins=50, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(MAX_LEN, color='red', linestyle='--', linewidth=1.2, label=f'max_length={MAX_LEN}')
ax.axvline(np.mean(total_tokens), color='orange', linestyle='--', linewidth=1.2, label=f'mean={np.mean(total_tokens):.0f}')
ax.set_xlabel('Token count')
ax.set_ylabel('Count')
ax.set_title('Full Example Token Lengths')
ax.legend(fontsize=9)

# Per-segment stacked view (mean)
ax = axes[1]
segments = ['question', 'reasoning', 'answer']
means = [np.mean(q_tokens), np.mean(r_tokens), np.mean(a_tokens)]
colors = ['#4C72B0', '#DD8452', '#55A868']
bars = ax.bar(segments, means, color=colors, edgecolor='white')
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Mean token count')
ax.set_title('Mean Tokens per Segment')

# CDF of total token lengths
ax = axes[2]
sorted_tokens = np.sort(total_tokens)
cdf = np.arange(1, len(sorted_tokens)+1) / len(sorted_tokens)
ax.plot(sorted_tokens, cdf, color='steelblue')
ax.axvline(MAX_LEN, color='red', linestyle='--', linewidth=1.2, label=f'max_length={MAX_LEN}')
p95_val = np.percentile(total_tokens, 95)
ax.axvline(p95_val, color='green', linestyle=':', linewidth=1.2, label=f'p95={p95_val:.0f}')
ax.set_xlabel('Token count')
ax.set_ylabel('Cumulative fraction')
ax.set_title('CDF of Token Lengths')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('token_distributions.png', bbox_inches='tight')
plt.show()
print("Saved token_distributions.png")

In [ ]:
# Word-level vocabulary (question text, before tokenization)
all_words = []
for r in records:
    all_words.extend(re.findall(r"[a-zA-Z']+", r['question'].lower()))
    all_words.extend(re.findall(r"[a-zA-Z']+", r['reasoning'].lower()))

word_freq = Counter(all_words)
total_words = sum(word_freq.values())
unique_words = len(word_freq)

print("=== Word-level vocabulary (raw text) ===")
print(f"  Total word tokens : {total_words:,}")
print(f"  Unique words      : {unique_words:,}")
print(f"  Type-token ratio  : {unique_words/total_words:.4f}")

# Coverage by top-N words
sorted_words = word_freq.most_common()
cumulative = 0
print("\n  Coverage by top-N words:")
for n in [50, 100, 500, 1000]:
    coverage = sum(c for _, c in sorted_words[:n]) / total_words
    print(f"    top {n:5d} : {coverage*100:.1f}%")

print("\n  Top 30 words:")
for word, count in sorted_words[:30]:
    print(f"    {word:<20} {count:6,}")

In [ ]:
# Number / operator statistics
all_numbers_q = []
all_numbers_r = []
for r in records:
    all_numbers_q.extend(re.findall(r'\d+\.?\d*', r['question']))
    all_numbers_r.extend(re.findall(r'\d+\.?\d*', r['reasoning']))

print(f"Distinct numbers in questions  : {len(set(all_numbers_q)):,}")
print(f"Distinct numbers in reasoning  : {len(set(all_numbers_r)):,}")
print(f"Total numeric tokens (questions): {len(all_numbers_q):,}")
print(f"Total numeric tokens (reasoning): {len(all_numbers_r):,}")

In [ ]:
if tokenizer is not None:
    vocab_size = tokenizer.vocab_size
    print(f"Custom tokenizer vocab size: {vocab_size:,}")

    # Token frequency across all training examples
    token_freq = Counter()
    for r in records:
        token_freq.update(tokenize(r['text']))

    used_tokens = len(token_freq)
    print(f"Tokens used (train set)      : {used_tokens:,} / {vocab_size:,} ({used_tokens/vocab_size*100:.1f}%)")
    print(f"Total token count (train)    : {sum(token_freq.values()):,}")

    # Zipf plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    ax = axes[0]
    freqs = sorted(token_freq.values(), reverse=True)
    ranks = np.arange(1, len(freqs)+1)
    ax.loglog(ranks, freqs, '.', markersize=2, alpha=0.5, color='steelblue')
    ax.set_xlabel('Token rank (log)')
    ax.set_ylabel('Frequency (log)')
    ax.set_title('Token Frequency (Zipf plot)')
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    cumfreq = np.cumsum(freqs) / sum(freqs)
    coverage_50 = np.searchsorted(cumfreq, 0.50) + 1
    coverage_90 = np.searchsorted(cumfreq, 0.90) + 1
    coverage_99 = np.searchsorted(cumfreq, 0.99) + 1
    ax.semilogx(ranks, cumfreq, color='steelblue')
    for pct, n in [(0.50, coverage_50), (0.90, coverage_90), (0.99, coverage_99)]:
        ax.axhline(pct, color='red', linestyle=':', linewidth=0.8)
        ax.text(n*1.1, pct+0.01, f'{pct*100:.0f}% @ rank {n}', fontsize=8)
    ax.set_xlabel('Number of top tokens (log)')
    ax.set_ylabel('Cumulative frequency')
    ax.set_title('Cumulative Token Coverage')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('vocab_statistics.png', bbox_inches='tight')
    plt.show()
    print(f"\n  50% coverage : top {coverage_50:,} tokens")
    print(f"  90% coverage : top {coverage_90:,} tokens")
    print(f"  99% coverage : top {coverage_99:,} tokens")
else:
    print("Skipping tokenizer-level vocab stats (tokenizer not loaded).")

In [ ]:
# Answer value distribution
def parse_answer(ans_str):
    ans_str = ans_str.strip().replace(',', '')
    try:
        return float(ans_str)
    except ValueError:
        return None

answer_vals = [parse_answer(r['answer']) for r in records]
answer_vals_clean = [v for v in answer_vals if v is not None]

print("=== Final answer value distribution ===")
print(f"  Numeric answers : {len(answer_vals_clean):,} / {len(records):,}")
print(f"  min  : {min(answer_vals_clean):.2f}")
print(f"  max  : {max(answer_vals_clean):.2f}")
print(f"  mean : {np.mean(answer_vals_clean):.2f}")
print(f"  median: {np.median(answer_vals_clean):.2f}")

neg_count = sum(1 for v in answer_vals_clean if v < 0)
print(f"\n  Negative answers : {neg_count} ({neg_count/len(answer_vals_clean)*100:.1f}%)")

In [ ]:
# Question topic diversity via first-word/bigram analysis
first_words = Counter()
for r in records:
    words = re.findall(r"[a-zA-Z']+", r['question'].lower())
    if words:
        first_words[words[0]] += 1

print("=== Top question-opening words (proxy for topic diversity) ===")
for word, count in first_words.most_common(20):
    print(f"  {word:<20} {count:5,} ({count/len(records)*100:.1f}%)")

In [ ]:
# Question length distribution
q_word_counts = [len(re.findall(r"[a-zA-Z0-9']+", r['question'])) for r in records]

print("=== Question word counts ===")
print(f"  min    : {min(q_word_counts)}")
print(f"  max    : {max(q_word_counts)}")
print(f"  mean   : {np.mean(q_word_counts):.1f}")
print(f"  median : {np.median(q_word_counts):.1f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.hist(q_word_counts, bins=40, color='#55A868', edgecolor='white', linewidth=0.4)
ax.axvline(np.mean(q_word_counts), color='red', linestyle='--', linewidth=1.2,
           label=f'mean={np.mean(q_word_counts):.1f}')
ax.set_xlabel('Word count')
ax.set_ylabel('Count')
ax.set_title('Question Length (words)')
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(answer_vals_clean, bins=60, color='#C44E52', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Answer value')
ax.set_ylabel('Count')
ax.set_title('Final Answer Value Distribution')
ax.set_xlim(-50, np.percentile(answer_vals_clean, 99))

ax = axes[2]
top_n = 15
fw_items = first_words.most_common(top_n)
words_sorted = [w for w, _ in fw_items]
counts_sorted = [c for _, c in fw_items]
ax.barh(words_sorted[::-1], counts_sorted[::-1], color='#8172B2', edgecolor='white')
ax.set_xlabel('Count')
ax.set_title(f'Top {top_n} Question-Opening Words')

plt.tight_layout()
plt.savefig('diversity_metrics.png', bbox_inches='tight')
plt.show()
print("Saved diversity_metrics.png")

In [ ]:
summary = {
    'Dataset': 'Arithmetic',
    'Train examples': f"{len(records):,}",
    'Test examples': f"{len(test_data):,}",
    '': '',
    'Token counts (full example)': '',
    '  min / mean / median / max': f"{min(total_tokens)} / {np.mean(total_tokens):.1f} / {np.median(total_tokens):.1f} / {max(total_tokens)}",
    '  p95 / p99': f"{np.percentile(total_tokens, 95):.0f} / {np.percentile(total_tokens, 99):.0f}",
    f'  Truncated at max_length={MAX_LEN}': f"{truncated} ({truncated/len(records)*100:.1f}%)",
    ' ': '',
    '  ': '',
    'Vocabulary (raw word-level)': '',
    '  Unique words': f"{unique_words:,}",
    '  Type-token ratio': f"{unique_words/total_words:.4f}",
}

if tokenizer is not None:
    summary['Tokenizer vocab size'] = f"{tokenizer.vocab_size:,}"
    summary['Tokens used (train set)'] = f"{used_tokens:,} ({used_tokens/vocab_size*100:.1f}%)"


for k, v in summary.items():
    if v == '':
        print()
    elif v == '':
        print()
    else:
        print(f"  {k:<45} {v}")


## Training Loop

In [ ]:
import os; os.environ['OUTPUT_DIR'] = SAVE_DIR

!python training/train.py